In [11]:
import kagglehub
path = kagglehub.dataset_download("tongpython/cat-and-dog")

Using Colab cache for faster access to the 'cat-and-dog' dataset.


In [12]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torch.optim as optim

In [13]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.ImageFolder(
    root=path + "/training_set/training_set",
    transform=transform
)

testset = torchvision.datasets.ImageFolder(
    root=path + "/test_set/test_set",
    transform=transform
)

trainloader = DataLoader(trainset , batch_size = 64 , shuffle=True)
testloader = DataLoader(testset , batch_size = 64)

In [14]:
class CNN(nn.Module):
  def __init__(self):
    super(CNN,self).__init__()

    self.conv_layers = nn.Sequential(
        nn.Conv2d(3,32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2 , 2),


        nn.Conv2d(32,64, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2 , 2),

        nn.Conv2d(64,128, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2 , 2)
    )

    self.fc_layers = nn.Sequential(
        nn.Linear(4*4*128 , 256),
        nn.ReLU(),

        nn.Linear(256 , 2)
    )

  def forward(self,x):
    x = self.conv_layers(x)
    x = x.view(x.size(0),-1)
    x = self.fc_layers(x)

    return x


In [15]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [16]:
for epoch in range(5):
    running_loss = 0.0

    model.train()

    for images, labels in trainloader:

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(trainloader)

    print(f"Epoch {epoch+1}/5 & Loss: {epoch_loss:.6f}")

Epoch 1/10 & Loss: 0.646928
Epoch 2/10 & Loss: 0.555733
Epoch 3/10 & Loss: 0.500308
Epoch 4/10 & Loss: 0.450646
Epoch 5/10 & Loss: 0.402424


In [17]:
correct = 0
total = 0

model.eval()

with torch.no_grad():
    for images, labels in trainloader:
        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f"Training Accuracy: {accuracy:.2f}%")

Training Accuracy: 80.81%


In [18]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in testloader:
        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Test Accuracy:", 100 * correct / total)

Test Accuracy: 75.13593672763223
